# Data Science · Woche 7 · Praktikum: Seite 2 einer Webseite

**B.Sc. Wirtschaftsinformatik · Hochschule Ruhr West · Do 19.11.2026**

**Die Frage (Olist):** Welche Bestellungen kommen zu spät – und was heißt das für die Bewertung?

Heute bauen Sie die Seite 2 einer Webseite, die Seite mit den Erkenntnissen: oben ein Filter, darunter Grafiken. Genau
diese Seite braucht auch Ihr Projekt. Das geht in zwei Teilen:

| Teil | Wo | Was |
|---|---|---|
| A · Aufgabe 1–9 | hier in Colab | Daten lesen, filtern, Grafiken mit Plotly – Zelle für Zelle, jede mit Check |
| B · Aufgabe 10 | auf Ihrem Laptop | dieselben Zeilen in eine Streamlit-Datei: Aus dem Notebook wird eine Webseite |

| Wann | Was |
|---|---|
| im Praktikum | gemeinsam am Beamer: der Fahrplan, dann Aufgabe 1 und 2 |
| zu Hause | Aufgabe 3 bis 10 in Ihrem Tempo, dann ⭐ 11 mit Ihren Daten. Die Musterlösung steht in Moodle – zum Vergleichen und wenn Sie festhängen. |

**Fahrplan: alle Aufgaben auf einen Blick**

| Aufgabe | Problem | Was Sie üben | Python | Im Projekt |
|---|---|---|---|---|
| 1 | Die Bestellungen liegen in `projekt.db` | Datenbank lesen | `sqlite3.connect` · `pd.read_sql` | alle Karten: Ihre Webseite liest `projekt.db` |
| 2 | Wer liest, will eine Gruppe ansehen | Filter: eine Variable wählt die Gruppe | `df[df[spalte] == wahl]` | alle: Bezirk, Beruf, Hotel, Jahr … |
| 3 | Wo kommen die meisten zu spät? | **Vergleich:** Anteil je Kategorie, liegende Balken, sortiert, Akzentfarbe, Titel aus den Daten | `groupby` · `px.bar(orientation='h')` · f-String | Karte 1 Jahreszeit · 2 Beruf · 4 Bezirk |
| 4 | Gilt das in jedem Staat? | **Vergleich** mit zwei Kategorien: gruppierte Balken | `groupby([a, b])` · `px.bar(color=…, barmode='group')` | Karte 3 Hotel und Kanal |
| 5 | Wie lange dauert eine Lieferung? | **Verteilung:** Histogramm, Klassenzahl, Ausreißer am Rand | `px.histogram(nbins=…)` · `clip` | alle: `cnt`, `balance`, `lead_time`, `price` |
| 6 | Kostet ein schweres Paket mehr Fracht? | **Zusammenhang:** viele Punkte, durchsichtig | `px.scatter(opacity=…)` | Karte 1 `temp`–`cnt` · 4 `accommodates`–`price` |
| 7 | Was kostet eine Verspätung an Sternen? | **Zusammenhang:** erst je Wert mitteln, dann zeichnen | `groupby` · `between` · `px.scatter` | Karte 2 Euribor–Quote · 3 `lead_time`–Quote |
| 8 | Eine überladene Grafik | aufräumen: Form, Farbe, Deko, Titel; vorher – nachher | `px.bar(orientation='h')` · `update_traces` | alle |
| 9 | Welche Diagramme gibt es noch? | ausprobieren, was welches Diagramm zeigt | `px.box` · `px.violin` · `px.density_heatmap` · … | Ideen für Ihre drei Grafiken |
| 10 | Aus dem Notebook wird eine Webseite | dieselben Zeilen in Streamlit; der Filter wird ein Auswahlfeld | `st.dataframe` · `st.selectbox` · `st.slider` · `st.plotly_chart` | Schritt 6: Ihre Seite 2 |
| ⭐ 11 | Ihre Daten | dasselbe in Ihrem `3_app.py` | Gerüst in `fuer_ihr_projekt/` | Schritt 6 und 8 |

**So ist jede Aufgabe gebaut:** 📖 worum es geht und warum → **Jetzt Sie** in der Code-Zelle (wo `None` steht, gehört
Ihr Ergebnis hin) → **Check** unter der Zelle: ✓ richtig · ✗ prüfen (mit dem erwarteten Wert) · … noch offen →
💡 **Tipp** zum Aufklappen. Die Aufgaben bauen aufeinander auf. Hängen Sie fest, nehmen Sie die Zeilen aus der
Musterlösung und machen Sie weiter.

## 0 · Setup (1 Minute)

**Datei → Kopie in Drive speichern**, dann die Zelle unten ausführen. Sie legt die Datenbank `projekt.db` in den
Ordner `app_daten` – genau so liegt sie in Teil B auf Ihrem Laptop. Erwartet: `app_daten/projekt.db: bereit (9.0 MB)`,
darunter `Setup ok`.

In [ ]:
import os
import sqlite3                      # Datenbanken im SQLite-Format
import urllib.request

import pandas as pd                 # Tabellen
import plotly.express as px         # Grafiken

GRAU = "#b4b4b4"        # alles, was nur zum Vergleich da ist
ROT = "#b03a2e"         # das, worauf es ankommt


def zahl(x, stellen=0):
    """Zahl im deutschen Format: 96470 → '96 470', 0.214 → '0,2'"""
    return f"{x:,.{stellen}f}".replace(",", " ").replace(".", ",")


os.makedirs("app_daten", exist_ok=True)
if not os.path.exists("app_daten/projekt.db"):                 # in Colab: aus dem Kurs-Repository holen
    urllib.request.urlretrieve("https://raw.githubusercontent.com/ZhuosHub/Data_Science_HRW_Praktikum/main/W7/app_daten/projekt.db",
                               "app_daten/projekt.db")
print(f"app_daten/projekt.db: bereit ({os.path.getsize('app_daten/projekt.db') / 1e6:.1f} MB)")
print("Setup ok")

---
# Teil A · In Colab: Daten, Filter, Grafiken

## Aufgabe 1 · Daten lesen

**📖 Worum es geht.** Ihre Webseite liest ihre Daten aus `app_daten/projekt.db` – so wie Ihre Projekt-Webseite aus
Ihrer `projekt.db` (Woche 6). Darin steht die Tabelle `bestellungen`: eine Zeile je gelieferter Bestellung, mit
Bundesstaat (`staat_kunde`), Produktkategorie (`kategorie`), Gewicht, Fracht, Lieferdauer in Tagen,
`tage_nach_termin` (negativ: früher als zugesagt), `spaet` (1 = zu spät, 0 = pünktlich) und `note` (1 bis 5 Sterne).
Lesen Sie die ganze Tabelle. Alle Zellen danach brauchen sie.

In [ ]:
# ===== Jetzt Sie (1) =====
bestellungen = None   # TODO: Verbindung zu app_daten/projekt.db, Tabelle bestellungen ganz lesen

# --- Check ---
if bestellungen is None:
    print("… noch offen")
elif len(bestellungen) != 96470 or "spaet" not in bestellungen.columns:
    print(f"✗ prüfen: erwartet 96 470 Zeilen mit allen 13 Spalten, gefunden {zahl(len(bestellungen))} Zeilen "
          f"und {len(bestellungen.columns)} Spalten – SELECT * FROM bestellungen")
else:
    print("✓ richtig · 96 470 Bestellungen, eine Zeile je Bestellung")
    display(bestellungen.head())

<details><summary><b>💡 Tipp</b> · erst selbst versuchen, dann aufklappen</summary>
<p>`verbindung = sqlite3.connect("app_daten/projekt.db")`, dann `pd.read_sql("SELECT * FROM bestellungen", verbindung)` – wie in Woche 6 beim Laden.</p>
</details>

## Aufgabe 2 · Filter: eine Produktkategorie

**📖 Worum es geht.** Seite 2 Ihrer Webseite braucht einen Filter, mit dem man die Daten einschränkt – auf einen
Bezirk, einen Beruf, ein Hotel. Hier: eine Produktkategorie. Auf der Webseite wird das ein Auswahlfeld (Aufgabe 10);
hier im Notebook ist es eine Variable, `wahl`. Steht dort `"Alle"`, bleiben alle Bestellungen, sonst nur die einer
Kategorie, z. B. `"health_beauty"`, `"bed_bath_table"` oder `"sports_leisure"`. Alle Grafiken darunter rechnen mit
`daten`, der gefilterten Tabelle. Wählen Sie später eine Kategorie, führen Sie diese Zelle und alle darunter noch
einmal aus (Menü **Laufzeit**). Auf der Webseite geschieht das bei jeder Auswahl von selbst.

In [ ]:
# ===== Jetzt Sie (2) =====
wahl = "Alle"   # auf der Webseite das Auswahlfeld; zum Ausprobieren z. B. "health_beauty"
daten = None    # TODO: bei "Alle" alle Bestellungen, sonst nur die Zeilen dieser Kategorie

# --- Check ---
if daten is None:
    print("… noch offen – bis dahin rechnen die Aufgaben unten mit allen Bestellungen")
    daten = bestellungen
elif wahl == "Alle" and len(daten) != len(bestellungen):
    print(f"✗ prüfen: bei „Alle“ erwartet {zahl(len(bestellungen))} Zeilen, gefunden {zahl(len(daten))}")
elif wahl != "Alle" and not (len(daten) == (bestellungen["kategorie"] == wahl).sum()
                             and (daten["kategorie"] == wahl).all()):
    print(f"✗ prüfen: bei „{wahl}“ erwartet {zahl((bestellungen['kategorie'] == wahl).sum())} Zeilen, "
          f"nur diese Kategorie – gefunden {zahl(len(daten))}")
else:
    print(f"✓ richtig · {zahl(len(daten))} Bestellungen in den Grafiken unten")

<details><summary><b>💡 Tipp</b> · erst selbst versuchen, dann aufklappen</summary>
<p>Ein Ausdruck mit `if … else` in einer Zeile: `daten = bestellungen if wahl == "Alle" else bestellungen[bestellungen["kategorie"] == wahl]`.</p>
</details>

## Aufgabe 3 · Vergleich: Wo kommen die meisten Bestellungen zu spät?

**📖 Worum es geht.** Ein Wert je Kategorie → Balken (Vorlesung, Teil ①). Der Wert ist hier ein **Anteil**: Der
Mittelwert einer 0/1-Spalte wie `spaet` ist der Anteil der Einsen. So rechnen Sie auch Abschlussquote (Karte 2) oder
Stornoquote (Karte 3). Staaten mit weniger als 1 % der Bestellungen lassen Sie weg: Bei wenigen Bestellungen entscheidet
der Zufall. Für andere gestaltet (Teil ②): **liegende** Balken, **sortiert**, nur der oberste in der Akzentfarbe `ROT`,
die anderen `GRAU`, und ein **Titel, der das Ergebnis nennt**. Weil der Filter die Daten ändert, schreiben Sie den Titel
mit einem f-String aus den Daten.

In [ ]:
# ===== Jetzt Sie (3) =====
je_staat = None   # TODO: je staat_kunde Anzahl (count) und Anteil zu spät (mean) von spaet; nur Staaten mit
                  #       mindestens 1 % der Bestellungen; Spalte prozent = Anteil * 100; aufsteigend sortiert
fig3 = None       # TODO: liegende Balken; Titel nennt den obersten Staat und seinen Anteil; nur er in ROT

# --- Check ---
if fig3 is None:
    print("… noch offen")
else:
    ref = daten.groupby("staat_kunde")["spaet"].agg(["count", "mean"])
    ref = ref[ref["count"] >= len(daten) / 100]
    top = ref["mean"].idxmax()
    balken = fig3.data[0]
    farben = balken.marker.color if isinstance(balken.marker.color, (list, tuple)) else [balken.marker.color]
    if balken.type != "bar" or balken.orientation != "h":
        print("✗ prüfen: liegende Balken – px.bar(…, orientation='h'), x = Anteil, y = Staat")
    elif len(balken.y) != len(ref):
        print(f"✗ prüfen: erwartet {len(ref)} Staaten mit mindestens 1 % der Bestellungen, gezeichnet {len(balken.y)}")
    elif list(balken.x) != sorted(balken.x):
        print("✗ prüfen: sortiert? sort_values vor dem Zeichnen – Plotly zeichnet die erste Zeile unten")
    elif balken.y[-1] != top:
        print(f"✗ prüfen: ganz oben sollte {top} stehen ({zahl(ref['mean'].max() * 100)} %)")
    elif top not in (fig3.layout.title.text or ""):
        print(f"✗ prüfen: Der Titel nennt das Ergebnis – den Staat ganz oben ({top}) und seinen Anteil")
    elif not (len(farben) == len(balken.y) and farben[-1] == ROT and set(farben[:-1]) == {GRAU}):
        print("✗ prüfen: nur der oberste Balken in ROT, alle anderen GRAU – eine Liste mit einer Farbe je Balken")
    else:
        print("✓ richtig · Mit einer anderen Kategorie in Aufgabe 2 ändern sich Balken und Titel mit.")
    fig3.show()

<details><summary><b>💡 Tipp</b> · erst selbst versuchen, dann aufklappen</summary>
<p>`daten.groupby("staat_kunde")["spaet"].agg(["count", "mean"]).reset_index()`; dann `je_staat[je_staat["count"] >= len(daten) / 100]` und `sort_values("prozent")`. Der oberste Staat ist `je_staat.iloc[-1]`. Titel: `f"In {oben['staat_kunde']} kommen {zahl(oben['prozent'])} % … zu spät"`. Farben: `fig3.update_traces(marker_color=[ROT if s == oben["staat_kunde"] else GRAU for s in je_staat["staat_kunde"]])`.</p>
</details>

## Aufgabe 4 · Vergleich mit zwei Kategorien: Note bei pünktlicher und verspäteter Lieferung

**📖 Worum es geht.** Zwei Kategorien je Wert – hier Bundesstaat und pünktlich/zu spät, bei Karte 3 Hotel und
Buchungskanal – zeichnen Sie als **gruppierte** Balken: je Staat zwei Balken nebeneinander (Teil ①). Nehmen Sie die
sechs Staaten mit den meisten Bestellungen und je Gruppe die mittlere Note. Damit die Legende Wörter zeigt, bekommt
`spaet` lesbare Namen.

In [ ]:
# ===== Jetzt Sie (4) =====
sechs = daten["staat_kunde"].value_counts().index[:6]          # die sechs Staaten mit den meisten Bestellungen
je_gruppe = None   # TODO: nur diese sechs; je staat_kunde und spaet die mittlere note;
                   #       Spalte lieferung: 0 → "pünktlich", 1 → "zu spät"
fig4 = None        # TODO: gruppierte Balken, Farbe = lieferung; Titel nennt das Ergebnis

# --- Check ---
if fig4 is None:
    print("… noch offen")
else:
    namen = {spur.name for spur in fig4.data}
    if fig4.data[0].type != "bar" or fig4.layout.barmode != "group":
        print("✗ prüfen: gruppierte Balken – px.bar(…, color='lieferung', barmode='group')")
    elif not {"pünktlich", "zu spät"} <= namen:
        print(f"✗ prüfen: die Legende sollte „pünktlich“ und „zu spät“ zeigen, sie zeigt {sorted(namen)} – "
              f"map({{0: 'pünktlich', 1: 'zu spät'}})")
    elif len(set(fig4.data[0].x)) != len(sechs):
        print(f"✗ prüfen: erwartet die sechs Staaten {', '.join(sechs)}")
    else:
        print("✓ richtig · je Staat zwei Balken nebeneinander: Der Unterschied ist überall groß.")
    fig4.show()

<details><summary><b>💡 Tipp</b> · erst selbst versuchen, dann aufklappen</summary>
<p>`daten[daten["staat_kunde"].isin(sechs)].groupby(["staat_kunde", "spaet"])["note"].mean().reset_index()`, dann `je_gruppe["spaet"].map({0: "pünktlich", 1: "zu spät"})`. Zeichnen: `px.bar(je_gruppe, x="staat_kunde", y="note", color="lieferung", barmode="group")`.</p>
</details>

## Aufgabe 5 · Verteilung: Wie lange dauert eine Lieferung?

**📖 Worum es geht.** Eine Zahl, viele Werte → Histogramm (Teil ①). Wie viele Klassen die Form zeigen, probiert man aus:
hier mit der Variablen `klassen`, auf der Webseite mit einem Regler. Ein paar Lieferungen dauern über 100 Tage; sie
würden die Achse so strecken, dass man den Rest nicht sieht. Legen Sie alles über 60 Tage an den rechten Rand und sagen
Sie das in der Achsenbeschriftung. Lange Ausläufer haben auch `cnt`, `balance`, `lead_time` und `price`.

In [ ]:
# ===== Jetzt Sie (5) =====
klassen = 60     # auf der Webseite ein Regler; probieren Sie auch 5 und 100
fig5 = None      # TODO: Histogramm von lieferdauer_tage, alles über 60 am Rand; Titel nennt den Median

# --- Check ---
if fig5 is None:
    print("… noch offen")
else:
    werte = fig5.data[0].x if fig5.data[0].x is not None else []
    median = zahl(daten["lieferdauer_tage"].median())
    if fig5.data[0].type != "histogram":
        print("✗ prüfen: ein Histogramm – px.histogram")
    elif fig5.data[0].nbinsx != klassen:
        print("✗ prüfen: Die Klassenzahl kommt aus der Variablen – nbins=klassen")
    elif len(werte) == 0 or max(werte) > 60:
        print("✗ prüfen: Werte über 60 Tage an den Rand legen – .clip(upper=60)")
    elif median not in (fig5.layout.title.text or ""):
        print(f"✗ prüfen: Der Titel nennt den Median ({median} Tage) – .median() im f-String")
    else:
        print("✓ richtig · Probieren Sie klassen = 5 und 100: Bei 5 ist die Form weg, bei 100 wird sie zackig.")
    fig5.show()

<details><summary><b>💡 Tipp</b> · erst selbst versuchen, dann aufklappen</summary>
<p>`tage = daten["lieferdauer_tage"].clip(upper=60)`; `px.histogram(x=tage, nbins=klassen, title=…, labels={"x": "Lieferdauer in Tagen (über 60 am rechten Rand)"})`.</p>
</details>

## Aufgabe 6 · Zusammenhang: Kostet ein schweres Paket mehr Fracht?

**📖 Worum es geht.** Zwei Zahlen → Streudiagramm (Teil ①). Bei Zehntausenden Punkten decken sie sich; zeichnen Sie
sie **durchsichtig**, dann sind dichte Stellen dunkel. Nehmen Sie die Pakete bis 30 kg – die wenigen schwereren würden
die Achse strecken – und schreiben Sie das an die Achse. So auch Karte 1 (`temp` und `cnt`) und Karte 4
(`accommodates` und `price`).

In [ ]:
# ===== Jetzt Sie (6) =====
bis_30_kg = None   # TODO: nur Bestellungen mit gewicht_g bis 30 000
fig6 = None        # TODO: Streudiagramm gewicht_g – fracht_brl, durchsichtig; Achsen mit Einheit

# --- Check ---
if fig6 is None:
    print("… noch offen")
else:
    punkte = fig6.data[0]
    deckkraft = punkte.marker.opacity if punkte.marker.opacity is not None else punkte.opacity
    if not punkte.type.startswith("scatter") or punkte.mode not in (None, "markers"):
        print("✗ prüfen: ein Streudiagramm – px.scatter")
    elif deckkraft is None or deckkraft >= 1:
        print("✗ prüfen: durchsichtig zeichnen – opacity=0.1")
    elif max(punkte.x) > 30000:
        print("✗ prüfen: nur Pakete bis 30 kg – daten[daten['gewicht_g'] <= 30000]")
    else:
        print("✓ richtig · Dunkle Stellen: Dort liegen viele Pakete übereinander.")
    fig6.show()

<details><summary><b>💡 Tipp</b> · erst selbst versuchen, dann aufklappen</summary>
<p>`bis_30_kg = daten[daten["gewicht_g"] <= 30000]`; `px.scatter(bis_30_kg, x="gewicht_g", y="fracht_brl", opacity=0.1, title=…, labels={…})`.</p>
</details>

## Aufgabe 7 · Zusammenhang: Was kostet eine Verspätung an Sternen?

**📖 Worum es geht.** Die Note ist nur 1 bis 5 Sterne: Ein Streudiagramm aller Bestellungen wäre fünf Streifen. Rechnen
Sie deshalb **erst je Wert** von `tage_nach_termin` die mittlere Note und zeichnen dann einen Punkt je Tag – nur von 30
Tagen vorher bis 30 Tage danach und nur Tage mit mindestens 10 Bestellungen, sonst entscheidet der Zufall. Genauso
Karte 2 (je Monat: Euribor und Abschlussquote) und Karte 3 (je Vorlaufzeit: Stornoquote).

In [ ]:
# ===== Jetzt Sie (7) =====
je_tag = None   # TODO: je tage_nach_termin Anzahl und Mittel der note; nur -30 bis 30 und mindestens 10 Bestellungen
fig7 = None     # TODO: ein Punkt je Tag; Achsen mit Einheit; Titel nennt das Ergebnis

# --- Check ---
if fig7 is None:
    print("… noch offen")
else:
    punkte = fig7.data[0]
    erwartet = daten.groupby("tage_nach_termin")["note"].agg(["count", "mean"]).reset_index()
    erwartet = erwartet[erwartet["tage_nach_termin"].between(-30, 30) & (erwartet["count"] >= 10)]
    if not punkte.type.startswith("scatter"):
        print("✗ prüfen: ein Streudiagramm – px.scatter")
    elif len(punkte.x) != len(erwartet):
        print(f"✗ prüfen: erwartet {len(erwartet)} Punkte – je Tag einer, nur -30 bis 30, mindestens 10 Bestellungen")
    elif min(punkte.y) < 1 or max(punkte.y) > 5:
        print("✗ prüfen: y ist die mittlere Note je Tag, also zwischen 1 und 5")
    else:
        print("✓ richtig · Bis zum Termin bleibt die Note hoch, danach fällt sie.")
    fig7.show()

<details><summary><b>💡 Tipp</b> · erst selbst versuchen, dann aufklappen</summary>
<p>`daten.groupby("tage_nach_termin")["note"].agg(["count", "mean"]).reset_index()`; dann `je_tag[je_tag["tage_nach_termin"].between(-30, 30) & (je_tag["count"] >= 10)]`; zeichnen mit `px.scatter(je_tag, x="tage_nach_termin", y="mean", …)`.</p>
</details>

## Aufgabe 8 · Aufräumen: vorher – nachher

**📖 Worum es geht.** Die Zelle unten zeigt eine Grafik, wie sie oft aussieht: Die Namen stehen schräg, jede Kategorie
hat eine eigene Farbe und dazu eine Legende, die Zahlen haben zwei Nachkommastellen, der Hintergrund ist bunt, der Titel
sagt nichts. Bauen Sie darunter dieselbe Aussage für andere (Teil ②): liegende, sortierte Balken in **einer** Farbe,
ohne Legende und bunten Hintergrund, mit einem Titel, der das Ergebnis nennt. Hier ohne Filter: alle Bestellungen.

In [ ]:
# ===== vorher =====
zehn = bestellungen["kategorie"].value_counts().head(10).reset_index()     # Spalten kategorie und count
vorher = px.bar(zehn, x="kategorie", y="count", color="kategorie", text_auto=".2f", title="Diagramm 1")
vorher.update_layout(plot_bgcolor="#fff3b0", paper_bgcolor="#d6eaff", font_family="Comic Sans MS")
vorher.update_traces(marker_line_color="black", marker_line_width=3)
vorher.update_xaxes(tickangle=60)
vorher.show()

In [ ]:
# ===== Jetzt Sie (8) =====
nachher = None   # TODO: dieselben zehn Kategorien, aufgeräumt

# --- Check ---
if nachher is None:
    print("… noch offen")
else:
    balken = nachher.data[0]
    if len(nachher.data) != 1:
        print("✗ prüfen: eine Farbe, keine Legende – ohne color=…")
    elif balken.orientation != "h" or list(balken.x) != sorted(balken.x):
        print("✗ prüfen: liegende Balken, sortiert – orientation='h', sort_values")
    elif nachher.layout.plot_bgcolor or nachher.layout.font.family:
        print("✗ prüfen: ohne bunten Hintergrund und Sonderschrift")
    elif (nachher.layout.title.text or "Diagramm 1") == "Diagramm 1":
        print("✗ prüfen: Der Titel nennt das Ergebnis")
    else:
        print("✓ richtig · gleiche Daten, eine Aussage")
    nachher.show()

<details><summary><b>💡 Tipp</b> · erst selbst versuchen, dann aufklappen</summary>
<p>`aufsteigend = zehn.sort_values("count")`; `px.bar(aufsteigend, x="count", y="kategorie", orientation="h", title=…, labels={…})`; eine Farbe: `nachher.update_traces(marker_color=GRAU)`.</p>
</details>

## Aufgabe 9 · Andere Diagramme ausprobieren

**📖 Worum es geht.** Dieselben Bestellungen lassen sich noch ganz anders zeigen – mit den anderen Diagrammen aus Teil ①.
Der Boxplot ist schon fertig. Ergänzen Sie die anderen fünf, je eine Zeile mit `px`, und sehen Sie sich an, was jedes
zeigt und was es versteckt. Vielleicht passt eines davon besser zu Ihrer Frage.

In [ ]:
# vorbereitet: die sechs Staaten aus Aufgabe 4, Lieferungen bis 60 Tage, Bestellungen je Monat, leichte Pakete
sechs_staaten = daten[daten["staat_kunde"].isin(daten["staat_kunde"].value_counts().index[:6])].copy()
sechs_staaten["lieferung"] = sechs_staaten["spaet"].map({0: "pünktlich", 1: "zu spät"})
bis_60_tage = sechs_staaten[sechs_staaten["lieferdauer_tage"] <= 60]
je_monat = daten.groupby(daten["kaufdatum"].str[:7]).size().reset_index(name="bestellungen")
leichte = daten[(daten["gewicht_g"] <= 5000) & (daten["fracht_brl"] <= 60)]   # der dichte Teil: bis 5 kg, bis 60 R$

# ===== vorgemacht: Boxplot – fünf Zahlen je Gruppe, was dazwischen liegt, fehlt =====
boxplot = px.box(bis_60_tage, x="staat_kunde", y="lieferdauer_tage",
                 labels={"staat_kunde": "Bundesstaat", "lieferdauer_tage": "Lieferdauer in Tagen (bis 60)"})
boxplot.show()

In [ ]:
# ===== Jetzt Sie (9) =====
violine = None     # TODO: px.violin mit denselben Spalten wie der Boxplot, box=True
heatmap = None     # TODO: px.density_heatmap(sechs_staaten, x="kaufmonat", y="staat_kunde", z="spaet", …)
gestapelt = None   # TODO: px.histogram(sechs_staaten, y="staat_kunde", color="lieferung", …)
hist_2d = None     # TODO: px.density_heatmap(leichte, x="gewicht_g", y="fracht_brl", nbinsx=50, nbinsy=40, …)
linie = None       # TODO: px.line(je_monat, x="kaufdatum", y="bestellungen")

# --- Check ---
for name, fig, typ, zeigt in [
        ("Violine", violine, "violin", "Boxplot + Dichtekurve: Wo die Violine breit ist, liegen viele Werte"),
        ("Heatmap", heatmap, "histogram2d", "zwei Kategorien je Wert, die Zahl als Farbe: große Muster"),
        ("Gestapelte Balken", gestapelt, "histogram", "Anteile mehrerer Gruppen vergleichen"),
        ("2D-Histogramm", hist_2d, "histogram2d", "viele Punkte: je Feld gezählt, die Anzahl als Farbe"),
        ("Linie", linie, "scatter", "eine Zahl über die Zeit: Verlauf, Trend")]:
    if fig is None:
        print(f"… noch offen: {name}")
    elif fig.data[0].type != typ:
        print(f"✗ prüfen: {name} – erwartet ein Diagramm vom Typ {typ}, gezeichnet {fig.data[0].type}")
    else:
        print(f"✓ {name} zeigt: {zeigt}")
        fig.show()

<details><summary><b>💡 Tipp</b> · erst selbst versuchen, dann aufklappen</summary>
<p>Alle `px`-Funktionen nehmen die Tabelle und die Spalten wie `px.box`. Violine: `box=True` zeichnet den Boxplot hinein. Heatmap mit Anteil: `histfunc="avg"` mittelt `spaet` je Feld, `nbinsx=12` gibt ein Feld je Monat, Farbskala `color_continuous_scale="Viridis"`. 2D-Histogramm: `nbinsx=50, nbinsy=40` teilt feiner ein. Gestapelt in %: `barnorm="percent"`.</p>
</details>

---
# Teil B · Auf Ihrem Laptop: Aus dem Notebook wird eine Webseite

## Aufgabe 10 · Seite 2 als Webseite: Filter und drei Grafiken

**📖 Worum es geht.** Bis hier haben Sie in Colab gerechnet und gezeichnet. Ihre Projekt-Webseite läuft aber mit
Streamlit: eine Python-Datei, die im Browser zur Webseite wird (Vorlesung). Ihre Zeilen aus Teil A bleiben gleich. Neu
sind nur die Befehle, die etwas auf die Seite bringen:

| hier in Colab | auf der Webseite |
|---|---|
| Die Zelle zeigt die Tabelle. | `st.dataframe(bestellungen.head(100))` |
| `wahl = "Alle"` – zum Ändern Zellen neu ausführen | `wahl = st.selectbox("Produktkategorie", ["Alle"] + kategorien)` |
| `klassen = 60` | `klassen = st.slider("Anzahl der Klassen", 5, 100, 60)` |
| Die Zelle zeigt die Grafik. | `st.plotly_chart(fig3)` |

1. Aus Moodle `W7_Praktikum.zip` herunterladen und entpacken. Darin liegen `W7_Webseite.py` und `app_daten/projekt.db`.
2. `W7_Webseite.py` im Editor öffnen (VS Code oder IDLE). An die Stellen `# ===== Jetzt Sie (…) =====` kopieren Sie Ihre
   Zeilen aus Aufgabe 1 und 2 und drei Grafiken: einen Vergleich (3 oder 4), eine Verteilung (5) und einen Zusammenhang
   (6 oder 7) – ohne die Checks. Dann die vier Befehle aus der Tabelle einsetzen.
3. Im Ordner ein Terminal öffnen und starten: `streamlit run W7_Webseite.py`. Nach jedem Speichern im Browser oben
   rechts **Rerun**.
4. Wählen Sie oben eine Kategorie: Alle drei Grafiken rechnen neu, ohne dass Sie eine Zelle ausführen. Bei jeder
   Auswahl läuft das Skript von oben nach unten neu – `wahl` ist einfach, was gerade ausgewählt ist.

## ⭐ Aufgabe 11 · Ihre Daten: Seite 2 in Ihrem `3_app.py`

**📖 Worum es geht.** Im Ordner `fuer_ihr_projekt/` liegt das Gerüst für `3_app.py`: drei Seiten, Seite 1 und die
Navigation sind fertig, Seite 3 kommt in Woche 10. Legen Sie es in Ihren Projektordner und füllen Sie Seite 2 mit Ihren
Daten: ein Filter und Ihre drei Grafiken – ein Vergleich, eine Verteilung, ein Zusammenhang, Vorschläge auf Ihrer
Projekt-Karte. Jede Grafik bekommt einen Titel, der das Ergebnis nennt.